# Expanded Q4 2025 AI/Tech Earnings Eval

This notebook runs or loads cached outputs for the expanded AI/Tech Q4 2025 earnings universe. Agent outputs are saved under `data/agent_outputs/q4_2025_ai_tech/` by agent type, so future runs can reuse them without calling the models again.


In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / "pyproject.toml").exists() else CURRENT_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
load_dotenv(REPO_ROOT / ".env", override=True)

from openclam.evaluation import q4_earnings_cache as q4
from openclam.agents.news_macro import news_macro_agent

print(f"Repo root: {REPO_ROOT}")
print(f"Cache root: {REPO_ROOT / q4.DEFAULT_CACHE_ROOT}")
print(f"VERTEX_PROJECT loaded: {bool(os.getenv('VERTEX_PROJECT') or os.getenv('GOOGLE_CLOUD_PROJECT'))}")
print(f"OPENAI_API_KEY loaded: {bool(os.getenv('OPENAI_API_KEY'))}")


Repo root: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory
Cache root: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech
VERTEX_PROJECT loaded: True
OPENAI_API_KEY loaded: False


## Configure Universe and Cache

Set `RUN_AGENTS=True` when you want to generate missing cached outputs. Keep `FORCE_RERUN=False` to reuse files that already exist.


In [2]:
import importlib
from openclam.evaluation import q4_earnings_cache as q4

q4 = importlib.reload(q4)

print(q4.__file__)
print(hasattr(q4, "q4_2025_combined_cio_advantage_df"))


/Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/src/openclam/evaluation/q4_earnings_cache.py
True


In [2]:
# Full combined Q4 2025 AI/Tech + CIO-advantage universe.
# This uses all tickers defined in q4_earnings_cache.py.
SELECTED_TICKERS = [
    # Mega-cap platforms
    "NVDA", "MSFT", "AMZN", "GOOGL", "META", "TSLA", "AAPL",

    # AI semis / supply chain
    "AMD", "AVGO", "MU", "TSM", "ASML",

    # AI infrastructure
    "ORCL", "ANET", "VRT", "DELL",

    # Power / data center
    "ETN", "CEG", "EQIX",

    # Software cloud
    "CRM",
]

earnings_df = q4.q4_2025_combined_cio_advantage_df(SELECTED_TICKERS)
earnings_df

CACHE_ROOT = REPO_ROOT / q4.DEFAULT_CACHE_ROOT
RUN_AGENTS = True
FORCE_RERUN = False  # keep False: cached names will not rerun; only missing tickers are generated

PRE_TRADING_DAYS = 7
POST_TRADING_DAYS = 7
LONG_POST_TRADING_DAYS = 30
NEUTRAL_BAND = 0.02

VERTEX_PROJECT = os.getenv("VERTEX_PROJECT") or os.getenv("GOOGLE_CLOUD_PROJECT")
VERTEX_LOCATION = os.getenv("VERTEX_LOCATION", "us-central1")
VERTEX_MODEL = os.getenv("VERTEX_MODEL", "gemini-2.5-flash")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
NEWS_MODEL = os.getenv("NEWS_MODEL", "gpt-5.4-nano")
LLM_PROVIDER = "vertex" if VERTEX_PROJECT else "openai"

print("Full universe size:", len(earnings_df))
earnings_df


Full universe size: 20


,ticker,company,earnings_date,bucket
0,AAPL,Apple,2026-01-29,mega_cap_platform
1,MSFT,Microsoft,2026-01-28,mega_cap_platform
2,GOOGL,Alphabet,2026-02-03,mega_cap_platform
3,AMZN,Amazon,2026-02-05,mega_cap_platform
4,META,Meta Platforms,2026-01-28,mega_cap_platform
5,TSLA,Tesla,2026-01-28,mega_cap_platform
6,NVDA,Nvidia,2026-02-25,mega_cap_platform
7,AMD,Advanced Micro Devices,2026-02-03,ai_semis
8,AVGO,Broadcom,2026-03-05,ai_semis
9,MU,Micron Technology,2025-12-17,ai_semis


## Build Price Eval Table


In [3]:
summary_df, paths_df = q4.build_price_tables(
    earnings_df=earnings_df,
    cache_root=CACHE_ROOT,
    pre_trading_days=PRE_TRADING_DAYS,
    post_trading_days=POST_TRADING_DAYS,
    long_post_trading_days=LONG_POST_TRADING_DAYS,
)

news_macro_agent.format_return_columns(summary_df)


,ticker,company,earnings_date,pre_7d_return,post_1d_return,post_3d_return,post_7d_return,post_30d_return,full_window_return,full_30d_window_return,...,qqq_post_7d_return,abnormal_vs_qqq,qqq_post_30d_return,abnormal_30d_vs_qqq,qqq_error,realized_direction_vs_qqq,realized_30d_direction_vs_qqq,long_horizon_trading_days,error,bucket
0,AAPL,Apple,2026-01-29,4.69%,0.46%,4.34%,6.43%,-3.07%,11.42%,1.48%,...,,,,,No close prices for QQQ from 2025-12-30 throug...,None,None,30.0,NaN,mega_cap_platform
1,MSFT,Microsoft,2026-01-28,,,,,,,,...,,,,,NaN,NaN,NaN,NaN,No close prices for MSFT from 2025-12-29 throu...,mega_cap_platform
2,GOOGL,Alphabet,2026-02-03,,,,,,,,...,,,,,NaN,NaN,NaN,NaN,No close prices for GOOGL from 2026-01-04 thro...,mega_cap_platform
3,AMZN,Amazon,2026-02-05,,,,,,,,...,,,,,NaN,NaN,NaN,NaN,No close prices for AMZN from 2026-01-06 throu...,mega_cap_platform
4,META,Meta Platforms,2026-01-28,,,,,,,,...,,,,,NaN,NaN,NaN,NaN,No close prices for META from 2025-12-29 throu...,mega_cap_platform
5,TSLA,Tesla,2026-01-28,-1.38%,-3.45%,-2.24%,-4.72%,-8.45%,-6.03%,-9.71%,...,-3.72%,-0.99%,-5.68%,-2.77%,NaN,down,down,30.0,NaN,mega_cap_platform
6,NVDA,Nvidia,2026-02-25,6.97%,-5.46%,-6.69%,-9.07%,-5.95%,-2.73%,0.61%,...,-2.75%,-6.33%,-0.93%,-5.02%,NaN,down,down,30.0,NaN,mega_cap_platform
7,AMD,Advanced Micro Devices,2026-02-03,-6.77%,-17.31%,-13.91%,-14.94%,-17.62%,-20.69%,-23.19%,...,-2.58%,-12.36%,-3.51%,-14.11%,NaN,down,down,30.0,NaN,ai_semis
8,AVGO,Broadcom,2026-03-05,2.24%,-0.69%,2.95%,-2.36%,22.42%,-0.18%,25.16%,...,-1.40%,-0.96%,6.69%,15.73%,NaN,down,up,30.0,NaN,ai_semis
9,MU,Micron Technology,2025-12-17,-8.67%,10.21%,22.65%,30.58%,94.21%,19.26%,77.38%,...,3.54%,27.04%,4.42%,89.79%,NaN,up,up,30.0,NaN,ai_semis


## Run or Load Cached Agent Outputs


In [6]:
if RUN_AGENTS:
    packets_by_ticker, errors_by_ticker = q4.run_q4_2025_universe_agents(
        earnings_df=earnings_df,
        cache_root=CACHE_ROOT,
        force=FORCE_RERUN,
        lookback_days=14,
        max_news=10,
        llm_provider=LLM_PROVIDER,
        vertex_project=VERTEX_PROJECT,
        vertex_location=VERTEX_LOCATION,
        vertex_model=VERTEX_MODEL,
        openai_model=OPENAI_MODEL,
        news_model=NEWS_MODEL,
    )
else:
    packets_by_ticker = q4.load_cached_packets_by_ticker(CACHE_ROOT)
    errors_by_ticker = {}

print("cached/available tickers:", sorted(packets_by_ticker))
errors_by_ticker


cached/available tickers: ['AAPL', 'AMD', 'AMZN', 'ANET', 'ASML', 'AVGO', 'CEG', 'CRM', 'DELL', 'EQIX', 'ETN', 'GOOGL', 'META', 'MSFT', 'MU', 'NVDA', 'ORCL', 'TSLA', 'TSM', 'VRT']


{'AAPL': {},
 'MSFT': {},
 'GOOGL': {},
 'AMZN': {},
 'META': {},
 'TSLA': {},
 'NVDA': {},
 'AMD': {},
 'AVGO': {},
 'MU': {},
 'TSM': {},
 'ASML': {},
 'ORCL': {},
 'DELL': {},
 'ANET': {},
 'VRT': {},
 'CRM': {},
 'ETN': {},
 'CEG': {},
 'EQIX': {}}

In [8]:
cio_eval, cio_results, cio_summary = q4.run_cached_cio_eval(
    summary_df,
    packets_by_ticker,
    cache_root=CACHE_ROOT,
    long_post_trading_days=LONG_POST_TRADING_DAYS,
    neutral_band=NEUTRAL_BAND,
    use_llm_debate=True,
    use_llm_decision=True,
    llm_provider=LLM_PROVIDER,
    debate_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
    decision_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
    vertex_project=VERTEX_PROJECT,
    vertex_location=VERTEX_LOCATION,
)


TypeError: run_cio_debate() got an unexpected keyword argument 'bucket'

In [19]:
from pathlib import Path
import pandas as pd

cio_dir = CACHE_ROOT / "cio"
cio_dir.mkdir(parents=True, exist_ok=True)

force_tickers = list(summary_df["ticker"].str.upper())

print("Force rerunning CIO tickers:", len(force_tickers))
print(force_tickers)

incremental_cio_evals = []
incremental_cio_results = {}

for ticker in force_tickers:
    print("\n" + "=" * 80)
    print(f"Force rerunning CIO for {ticker}...")
    print("=" * 80)

    one_summary_df = summary_df[
        summary_df["ticker"].str.upper() == ticker
    ].copy()

    one_packets = {
        ticker: packets_by_ticker.get(ticker, [])
    }

    if not one_packets[ticker]:
        print(f"Skipped {ticker}: no packets found.")
        continue

    try:
        one_cio_eval, one_cio_results, one_cio_summary = q4.run_cached_cio_eval(
            one_summary_df,
            one_packets,
            cache_root=CACHE_ROOT,
            long_post_trading_days=LONG_POST_TRADING_DAYS,
            neutral_band=NEUTRAL_BAND,
            use_llm_debate=True,
            use_llm_decision=True,
            llm_provider=LLM_PROVIDER,
            debate_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
            decision_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
            vertex_project=VERTEX_PROJECT,
            vertex_location=VERTEX_LOCATION,
        )

        incremental_cio_evals.append(one_cio_eval)
        incremental_cio_results[ticker] = one_cio_results.get(ticker)

        print(f"Overwrote CIO result for {ticker}: {cio_dir / f'{ticker}.json'}")

    except Exception as exc:
        print(f"Failed CIO for {ticker}: {repr(exc)}")


Force rerunning CIO tickers: 20
['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'TSLA', 'NVDA', 'AMD', 'AVGO', 'MU', 'TSM', 'ASML', 'ORCL', 'DELL', 'ANET', 'VRT', 'CRM', 'ETN', 'CEG', 'EQIX']

Force rerunning CIO for AAPL...
Overwrote CIO result for AAPL: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/cio/AAPL.json

Force rerunning CIO for MSFT...
Overwrote CIO result for MSFT: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/cio/MSFT.json

Force rerunning CIO for GOOGL...
Overwrote CIO result for GOOGL: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/cio/GOOGL.json

Force rerunning CIO for AMZN...
Overwrote CIO result for AMZN: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/cio/AMZN.json

Force rerunning CIO for META...
Overwrote CIO result for META: /Users/zhang/Github/OpenClam_Multi-Agent_Investment

## Run CIO Eval From Cached Packets


In [15]:
expected_tickers = set(summary_df["ticker"].str.upper())

cached_cio_files = {
    path.stem.upper()
    for path in (CACHE_ROOT / "cio").glob("*.json")
}

missing_cio_tickers = sorted(expected_tickers - cached_cio_files)

print("Missing CIO tickers:", missing_cio_tickers)

incremental_results = {}

for ticker in missing_cio_tickers:
    print("\n" + "=" * 80)
    print(f"Running CIO for {ticker}...")
    print("=" * 80)

    one_summary_df = summary_df[
        summary_df["ticker"].str.upper() == ticker
    ].copy()

    one_packets = {
        ticker: packets_by_ticker.get(ticker, [])
    }

    if not one_packets[ticker]:
        print(f"Skipped {ticker}: no packets found.")
        continue

    try:
        one_cio_eval, one_cio_results, one_cio_summary = q4.run_cached_cio_eval(
            one_summary_df,
            one_packets,
            cache_root=CACHE_ROOT,
            long_post_trading_days=LONG_POST_TRADING_DAYS,
            neutral_band=NEUTRAL_BAND,
            use_llm_debate=True,
            use_llm_decision=True,
            llm_provider=LLM_PROVIDER,
            debate_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
            decision_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
            vertex_project=VERTEX_PROJECT,
            vertex_location=VERTEX_LOCATION,
        )

        incremental_results[ticker] = one_cio_results.get(ticker)
        print(f"Saved CIO result for {ticker}: {CACHE_ROOT / 'cio' / f'{ticker}.json'}")

    except Exception as exc:
        print(f"Failed CIO for {ticker}: {repr(exc)}")


Missing CIO tickers: []


In [20]:
# Rebuild full cio_eval from all cached CIO workflows for the current summary_df universe

expected_tickers = set(summary_df["ticker"].str.upper())

cio_results = {
    path.stem.upper(): q4.load_json(path)
    for path in (CACHE_ROOT / "cio").glob("*.json")
    if path.stem.upper() in expected_tickers
}

print("Expected tickers:", len(expected_tickers))
print("Loaded cached CIO workflows:", len(cio_results))
print("Missing cached CIO workflows:", sorted(expected_tickers - set(cio_results)))

def stance_to_direction(stance):
    if stance == "Bullish":
        return "up"
    if stance == "Bearish":
        return "down"
    return None

def match_direction(predicted, realized, abnormal, neutral_band=0.02):
    if pd.isna(abnormal) or realized is None:
        return None
    abnormal = float(abnormal)
    if predicted is None:
        return abs(abnormal) <= neutral_band
    return predicted == realized

def match_reason(predicted, realized, abnormal, neutral_band=0.02):
    if pd.isna(abnormal) or realized is None:
        return "missing realized abnormal return"
    abnormal = float(abnormal)
    if predicted is None:
        if abs(abnormal) <= neutral_band:
            return "neutral matched: abnormal return stayed inside neutral band"
        return "neutral missed: abnormal return moved outside neutral band"
    return "matched" if predicted == realized else "missed"

long_abnormal_col = f"abnormal_{LONG_POST_TRADING_DAYS}d_vs_qqq"
long_direction_col = f"realized_{LONG_POST_TRADING_DAYS}d_direction_vs_qqq"

rows = []

for _, row in summary_df.iterrows():
    ticker = row["ticker"].upper()
    workflow = cio_results.get(ticker)
    base = row.to_dict()

    if not workflow:
        base.update({
            "cio_ready": False,
            "cio_short_term_stance": None,
            "cio_long_term_stance": None,
            "cio_action": None,
            "cio_confidence": None,
            "cio_debate_triggered": None,
            "cio_debate_conflict_level": None,
            "cio_reason": "missing cached CIO workflow",
            "cio_short_direction_match": None,
            "cio_short_direction_match_reason": "missing cached CIO workflow",
            "cio_long_direction_match": None,
            "cio_long_direction_match_reason": "missing cached CIO workflow",
        })
        rows.append(base)
        continue

    decision = workflow["final_decision"]
    debate = workflow["debate"]

    short_stance = decision["final_short_term_stance"]
    long_stance = decision["final_long_term_stance"]

    short_pred = stance_to_direction(short_stance)
    long_pred = stance_to_direction(long_stance)

    base.update({
        "cio_ready": True,
        "cio_short_term_stance": short_stance,
        "cio_long_term_stance": long_stance,
        "cio_action": decision.get("investment_action"),
        "cio_confidence": decision.get("confidence"),
        "cio_debate_triggered": debate.get("triggered"),
        "cio_debate_conflict_level": debate.get("trigger", {}).get("conflict_level"),
        "cio_reason": decision.get("reason_for_final_decision"),
        "cio_short_direction_match": match_direction(
            short_pred,
            row.get("realized_direction_vs_qqq"),
            row.get("abnormal_vs_qqq"),
            NEUTRAL_BAND,
        ),
        "cio_short_direction_match_reason": match_reason(
            short_pred,
            row.get("realized_direction_vs_qqq"),
            row.get("abnormal_vs_qqq"),
            NEUTRAL_BAND,
        ),
        "cio_long_direction_match": match_direction(
            long_pred,
            row.get(long_direction_col),
            row.get(long_abnormal_col),
            NEUTRAL_BAND,
        ),
        "cio_long_direction_match_reason": match_reason(
            long_pred,
            row.get(long_direction_col),
            row.get(long_abnormal_col),
            NEUTRAL_BAND,
        ),
    })

    rows.append(base)

cio_eval = pd.DataFrame(rows)

cio_summary = {
    "overall": {
        "cases": int(len(cio_eval)),
        "cio_ready_cases": int((cio_eval["cio_ready"] == True).sum()),
        "debate_trigger_rate": float((cio_eval["cio_debate_triggered"] == True).mean()),
        "cio_short_direction_match_evaluable": int(cio_eval["cio_short_direction_match"].notna().sum()),
        "cio_short_direction_match_matched": int((cio_eval["cio_short_direction_match"] == True).sum()),
        "cio_short_direction_match_accuracy": float(
            (cio_eval[cio_eval["cio_short_direction_match"].notna()]["cio_short_direction_match"] == True).mean()
        ),
        "cio_long_direction_match_evaluable": int(cio_eval["cio_long_direction_match"].notna().sum()),
        "cio_long_direction_match_matched": int((cio_eval["cio_long_direction_match"] == True).sum()),
        "cio_long_direction_match_accuracy": float(
            (cio_eval[cio_eval["cio_long_direction_match"].notna()]["cio_long_direction_match"] == True).mean()
        ),
    },
    "by_bucket": q4.summarize_cio_eval_by_bucket(cio_eval),
}

cio_eval.to_csv(CACHE_ROOT / "tables" / "cio_eval_combined_cache_aware.csv", index=False)
q4.save_json(CACHE_ROOT / "tables" / "cio_eval_combined_cache_aware_summary.json", cio_summary)

print(cio_summary)
news_macro_agent.format_return_columns(cio_eval)


Expected tickers: 20
Loaded cached CIO workflows: 20
Missing cached CIO workflows: []
{'overall': {'cases': 20, 'cio_ready_cases': 20, 'debate_trigger_rate': 0.45, 'cio_short_direction_match_evaluable': 15, 'cio_short_direction_match_matched': 4, 'cio_short_direction_match_accuracy': 0.26666666666666666, 'cio_long_direction_match_evaluable': 15, 'cio_long_direction_match_matched': 10, 'cio_long_direction_match_accuracy': 0.6666666666666666}, 'by_bucket': {'ai_infrastructure': {'cases': 4, 'debate_trigger_rate': 0.25, 'short_accuracy': 0.25, 'long_accuracy': 0.75, 'avg_confidence': 0.8}, 'ai_semis': {'cases': 5, 'debate_trigger_rate': 0.2, 'short_accuracy': 0.4, 'long_accuracy': 0.8, 'avg_confidence': 0.8088}, 'data_center_reit': {'cases': 1, 'debate_trigger_rate': 0.0, 'short_accuracy': 0.0, 'long_accuracy': 1.0, 'avg_confidence': 0.8}, 'mega_cap_platform': {'cases': 7, 'debate_trigger_rate': 0.7142857142857143, 'short_accuracy': 0.5, 'long_accuracy': 0.5, 'avg_confidence': 0.778571428

,ticker,company,earnings_date,pre_7d_return,post_1d_return,post_3d_return,post_7d_return,post_30d_return,full_window_return,full_30d_window_return,...,cio_long_term_stance,cio_action,cio_confidence,cio_debate_triggered,cio_debate_conflict_level,cio_reason,cio_short_direction_match,cio_short_direction_match_reason,cio_long_direction_match,cio_long_direction_match_reason
0,AAPL,Apple,2026-01-29,4.69%,0.46%,4.34%,6.43%,-3.07%,11.42%,1.48%,...,Neutral,Hold / Watch,0.850,True,high,The committee reached a unanimous Bullish shor...,None,missing realized abnormal return,None,missing realized abnormal return
1,MSFT,Microsoft,2026-01-28,,,,,,,,...,Bullish,Accumulate on weakness,0.750,False,low,The decision aligns with the deterministic gua...,None,missing realized abnormal return,None,missing realized abnormal return
2,GOOGL,Alphabet,2026-02-03,,,,,,,,...,Bullish,Wait for pullback,0.700,False,low,The final decision is to adopt a Bearish short...,None,missing realized abnormal return,None,missing realized abnormal return
3,AMZN,Amazon,2026-02-05,,,,,,,,...,Bullish,Wait for pullback to establish long position,0.850,True,high,The committee reached a high-conviction consen...,None,missing realized abnormal return,None,missing realized abnormal return
4,META,Meta Platforms,2026-01-28,,,,,,,,...,Bullish,Accumulate on weakness,0.800,True,high,The final decision is an override of the initi...,None,missing realized abnormal return,None,missing realized abnormal return
5,TSLA,Tesla,2026-01-28,-1.38%,-3.45%,-2.24%,-4.72%,-8.45%,-6.03%,-9.71%,...,Bearish,Short / Avoid,0.850,True,high,The decision is driven by the unanimous post-d...,True,matched,True,matched
6,NVDA,Nvidia,2026-02-25,6.97%,-5.46%,-6.69%,-9.07%,-5.95%,-2.73%,0.61%,...,Neutral,Hold / Monitor,0.650,True,high,CIO Override: I am overriding the deterministi...,False,missed,False,neutral missed: abnormal return moved outside ...
7,AMD,Advanced Micro Devices,2026-02-03,-6.77%,-17.31%,-13.91%,-14.94%,-17.62%,-20.69%,-23.19%,...,Bullish,Accumulate on Weakness,0.850,True,high,I am overriding the deterministic guardrail's ...,True,matched,False,missed
8,AVGO,Broadcom,2026-03-05,2.24%,-0.69%,2.95%,-2.36%,22.42%,-0.18%,25.16%,...,Bullish,Long,0.800,False,low,The committee is overriding the guardrail's 'S...,False,missed,True,matched
9,MU,Micron Technology,2025-12-17,-8.67%,10.21%,22.65%,30.58%,94.21%,19.26%,77.38%,...,Bullish,Buy,0.872,False,low,The committee reached a high-conviction bullis...,True,matched,True,matched


## Inspect Saved Outputs


In [14]:
ticker_to_inspect = "NVDA"
cached = q4.load_cached_ticker(ticker_to_inspect, CACHE_ROOT)
print("saved files:")
for key, value in q4.cached_ticker_paths(ticker_to_inspect, CACHE_ROOT).items():
    print(key, value)

packets = cached.get("packets") or []
packet_df = pd.DataFrame(packets)
display_cols = ["agent_name", "short_term_stance", "long_term_stance", "confidence", "stance_rationale"]
packet_df[[col for col in display_cols if col in packet_df.columns]]


saved files:
news_macro /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/news_macro/NVDA.json
market_technical /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/market_technical/NVDA.json
fundamental /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/fundamental/NVDA.json
packets /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/packets/NVDA.json
cio /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/cio/NVDA.json
context /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/contexts/NVDA.json
errors /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/errors/NVDA.json


,agent_name,short_term_stance,long_term_stance,confidence,stance_rationale
0,news_macro,Bullish,Bearish,0.65,The short-term stance is Bullish due to a dire...
1,fundamental,Neutral,Bullish,0.60,NVIDIA reported an exceptionally strong quarte...
2,market_technical,Neutral,Bullish,0.70,The primary tension is between the established...


In [15]:
# Debate trace for one ticker
workflow = cio_results.get(ticker_to_inspect) or q4.load_json(CACHE_ROOT / "cio" / f"{ticker_to_inspect}.json")
pd.DataFrame(workflow["debate"].get("debate_responses", []))


,response_type,agreement,disagreement,revised_view,evidence_needed,revised_short_term_stance,revised_long_term_stance,revised_confidence,revised_view_summary
0,revision,{'fundamental': 'I agree with the fundamental ...,{'fundamental': 'I disagree with the implicati...,"{'ticker': 'NVDA', 'company': 'Nvidia', 'agent...",[Management commentary from the earnings call ...,NaN,NaN,NaN,NaN
1,revised_agent_view,{'agreed_with_news_macro': 'The news_macro age...,{'disagreement_with_news_macro': 'While I now ...,NaN,[Quantitative analysis of AMD's potential mark...,Bullish,Bullish,0.80,{'confidence_rationale': 'Confidence has incre...
2,revision,{'news_macro': 'The evidence of a 'strong earn...,{'news_macro_stance': 'I disagree with the `lo...,NaN,[Post-earnings price action to confirm a break...,Bullish,Bullish,0.85,NaN


In [19]:
import pandas as pd

if "incremental_cio_evals" in globals() and incremental_cio_evals:
    cio_eval = pd.concat(incremental_cio_evals, ignore_index=True)
    cio_results = incremental_cio_results
else:
    cio_eval = pd.read_csv(CACHE_ROOT / "tables" / "cio_eval_selected.csv")
    cio_results = {}

cio_summary = {
    "overall": {
        "cases": int(len(cio_eval)),
        "cio_ready_cases": int((cio_eval["cio_ready"] == True).sum()),
        "debate_trigger_rate": float((cio_eval["cio_debate_triggered"] == True).mean()),
        "cio_short_direction_match_evaluable": int(cio_eval["cio_short_direction_match"].notna().sum()),
        "cio_short_direction_match_matched": int((cio_eval["cio_short_direction_match"] == True).sum()),
        "cio_short_direction_match_accuracy": float(
            (cio_eval[cio_eval["cio_short_direction_match"].notna()]["cio_short_direction_match"] == True).mean()
        ),
        "cio_long_direction_match_evaluable": int(cio_eval["cio_long_direction_match"].notna().sum()),
        "cio_long_direction_match_matched": int((cio_eval["cio_long_direction_match"] == True).sum()),
        "cio_long_direction_match_accuracy": float(
            (cio_eval[cio_eval["cio_long_direction_match"].notna()]["cio_long_direction_match"] == True).mean()
        ),
    },
    "by_bucket": q4.summarize_cio_eval_by_bucket(cio_eval),
}

cio_eval.to_csv(CACHE_ROOT / "tables" / "cio_eval_selected.csv", index=False)
q4.save_json(CACHE_ROOT / "tables" / "cio_eval_selected_summary.json", cio_summary)

news_macro_agent.format_return_columns(cio_eval)


,ticker,company,earnings_date,pre_7d_return,post_1d_return,post_3d_return,post_7d_return,post_30d_return,full_window_return,full_30d_window_return,...,cio_long_term_stance,cio_action,cio_confidence,cio_debate_triggered,cio_debate_conflict_level,cio_reason,cio_short_direction_match,cio_short_direction_match_reason,cio_long_direction_match,cio_long_direction_match_reason
0,EQIX,Equinix,2026-02-11,7.05%,10.41%,9.73%,9.00%,11.66%,16.69%,19.53%,...,Bullish,Accumulate on weakness,0.8,False,low,I am overriding the deterministic guardrail's ...,False,neutral missed: abnormal return moved outside ...,True,matched


# CIO vs single-agent and DeepResearch comparison

Earnings momentum is now part of the Market/Technical agent input, so the main comparison table focuses on model/agent outputs only:

- CIO committee synthesis
- single agents: news/macro, market/technical, fundamental
- optional DeepResearch labels if `deepresearch_labels.csv` is saved under `CACHE_ROOT / "tables"`


In [21]:
import pandas as pd

comparison_eval = q4.add_single_agent_baselines(
    cio_eval,
    packets_by_ticker,
    long_post_trading_days=LONG_POST_TRADING_DAYS,
    neutral_band=NEUTRAL_BAND,
)

strategy_prefixes = [
    "cio",
    "news_macro",
    "market_technical",
    "fundamental",
]

deepresearch_path = CACHE_ROOT / "tables" / "deepresearch_labels.csv"
if deepresearch_path.exists():
    deepresearch_df = pd.read_csv(deepresearch_path)
    comparison_eval = q4.add_deepresearch_baseline(
        comparison_eval,
        deepresearch_df,
        long_post_trading_days=LONG_POST_TRADING_DAYS,
        neutral_band=NEUTRAL_BAND,
    )
    strategy_prefixes.append("deepresearch")
else:
    q4.write_deepresearch_label_template(comparison_eval, deepresearch_path)
    print(f"DeepResearch template created: {deepresearch_path}")

strategy_summary = q4.build_strategy_comparison_table(comparison_eval, strategy_prefixes)
strategy_bucket_summary = q4.build_strategy_bucket_comparison(comparison_eval, strategy_prefixes)

comparison_eval.to_csv(CACHE_ROOT / "tables" / "agent_strategy_comparison_eval.csv", index=False)
strategy_summary.to_csv(CACHE_ROOT / "tables" / "agent_strategy_summary.csv", index=False)
strategy_bucket_summary.to_csv(CACHE_ROOT / "tables" / "agent_strategy_bucket_summary.csv", index=False)

strategy_summary


,strategy,short_evaluable,short_matched,short_accuracy,long_evaluable,long_matched,long_accuracy
0,cio,15,4,0.266667,15,10,0.666667
1,news_macro,15,5,0.333333,15,12,0.800000
2,market_technical,15,5,0.333333,15,9,0.600000
3,fundamental,15,2,0.133333,15,5,0.333333
4,deepresearch,15,2,0.133333,15,1,0.066667


In [22]:
strategy_bucket_summary


,bucket,cases,cio_short_accuracy,cio_long_accuracy,news_macro_short_accuracy,news_macro_long_accuracy,market_technical_short_accuracy,market_technical_long_accuracy,fundamental_short_accuracy,fundamental_long_accuracy,deepresearch_short_accuracy,deepresearch_long_accuracy
0,ai_infrastructure,4,0.25,0.75,0.5,0.75,0.25,0.75,0.0,0.25,0.0,0.0
1,ai_semis,5,0.40,0.80,0.4,0.80,0.40,0.80,0.2,0.60,0.2,0.2
2,data_center_reit,1,0.00,1.00,0.0,1.00,0.00,1.00,0.0,0.00,0.0,0.0
3,mega_cap_platform,7,0.50,0.50,0.5,1.00,0.00,0.00,0.5,0.00,0.5,0.0
4,power_infrastructure,2,0.00,0.50,0.0,0.50,1.00,0.50,0.0,0.50,0.0,0.0
5,software_cloud,1,0.00,0.00,0.0,1.00,0.00,0.00,0.0,0.00,0.0,0.0


In the selected 20-name demo universe, the CIO layer adds the clearest value in second-order AI supply-chain buckets, especially AI semiconductors and AI infrastructure. It is less effective for mega-cap platform stocks, where direct news/macro interpretation dominates. This supports the intended design: the CIO is most useful when investment implications require combining technical confirmation, fundamental quality, and second-order industry-chain reasoning.